In [2]:
import cv2
import mediapipe as mp
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# --- CONFIGURATION ---
# !!! IMPORTANT: SET THE PATH TO YOUR YOLOv8 CLASSIFICATION MODEL !!!
MODEL_PATH = "best.pt" # Your Ultralytics .pt file

IMAGE_PATH = r"D:\Courses\MyProjects\Static Sign Language\MediaPipe-yolo-app\MyHandDataSetMediaPipeLandmarksOnly\train\images\Baba.a2d2218a-45e7-11f0-aeb4-8c164508cf85.jpg"

print(f"Loading the Ultralytics model from: {MODEL_PATH}")
# This single line loads the model and prepares it for inference.
model = YOLO(MODEL_PATH)
print("Model loaded successfully.")
    
# --- Verify that files exist ---
if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model file not found at: {MODEL_PATH}")
if not os.path.exists(IMAGE_PATH):
    print(f"ERROR: Image file not found at: {IMAGE_PATH}")

Loading the Ultralytics model from: best.pt
Model loaded successfully.


In [4]:
# --- Load the YOLOv8 Model ---
try:
    model = YOLO(MODEL_PATH)
    class_names = model.names
    print("Ultralytics model loaded successfully.")
    print(f"Model classes: {class_names}")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None

# --- Initialize MediaPipe Hands ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,  # True is best for single images
    max_num_hands=2,
    min_detection_confidence=0.2
)
mp_drawing = mp.solutions.drawing_utils

print("MediaPipe Hands initialized.")

Ultralytics model loaded successfully.
Model classes: {0: 'baba', 1: 'book', 2: 'company', 3: 'grandfather', 4: 'mall', 5: 'mama', 6: 'melon', 7: 'mosque', 8: 'photograph', 9: 'salaam alaikum', 10: 'salam', 11: 'school', 12: 'university'}
MediaPipe Hands initialized.


In [5]:
if model:
    # --- Initialize MediaPipe and OpenCV ---
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        static_image_mode=False, max_num_hands=2,
        min_detection_confidence=0.2, min_tracking_confidence=0.5
    )
    mp_drawing = mp.solutions.drawing_utils

    cap = cv2.VideoCapture(0)

    # Get the names of the classes from the model
    class_names = model.names

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        display_frame = frame.copy()
        h, w, _ = frame.shape
        
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results_mp = hands.process(image_rgb)
        
        # Create a blank image to draw landmarks for model input
        model_input_image = np.ones((h, w, 3), dtype=np.uint8) * 255
        
        if results_mp.multi_hand_landmarks:
            for hand_landmarks in results_mp.multi_hand_landmarks:
                # Draw landmarks on the display frame (for us to see)
                mp_drawing.draw_landmarks(display_frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
                # Draw landmarks on the blank image (for the model to see)
                mp_drawing.draw_landmarks(
                    model_input_image, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=4),
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2)
                )

                # --- ULTRALYTICS PREDICTION ---
                # The 'predict' method handles all the preprocessing and inference.
                # verbose=False prevents it from printing logs for every frame.
                results_yolo = model.predict(model_input_image, verbose=False)
                
                # The result object contains all the information
                result = results_yolo[0]
                probs = result.probs  # Probs object for classification models

                if probs is not None:
                    # Get the index and confidence of the top prediction
                    top1_index = probs.top1
                    top1_confidence = probs.top1conf.item()

                    if top1_confidence > 0.75: # Confidence threshold
                        # Get the class name using the index
                        predicted_label = class_names[top1_index]
                        
                        # Get bounding box to position the text
                        x_coords = [lm.x for lm in hand_landmarks.landmark]
                        y_coords = [lm.y for lm in hand_landmarks.landmark]
                        x_min, y_min = int(min(x_coords) * w), int(min(y_coords) * h)
                        
                        text = f"Sign: {predicted_label} ({top1_confidence:.2f})"
                        cv2.putText(display_frame, text, (x_min, y_min - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
        
        # Display the main camera feed
        cv2.imshow('Real-Time Hand Sign Detection (Ultralytics)', display_frame)
        # Display what the model is "seeing" - this is great for debugging
        cv2.imshow('Model Input', model_input_image)
        
        key = cv2.waitKey(5) & 0xFF
        if key == ord('q') or key == 27: break

    print("Releasing resources...")
    cap.release()
    cv2.destroyAllWindows()
    hands.close()
else:
    print("Skipping real-time detection due to loading errors.")

Releasing resources...


In [ ]:
# --- Step 1: Import necessary libraries ---
import cv2
import numpy as np
import mediapipe as mp
import torch
from ultralytics import YOLO
import PIL.Image
from IPython.display import display, Image

# --- Step 2: Configuration ---

# Path to your trained YOLOv8 model
MODEL_PATH = 'best.pt' # <--- IMPORTANT: CHANGE THIS

# The image size your model was trained on
IMAGE_SIZE = 224

# A confidence threshold for displaying the prediction
# (e.g., only show the prediction if the model is > 70% confident)
CONFIDENCE_THRESHOLD = 0.70

# --- Step 3: Copy Helper Functions ---
# These are the same functions used to generate your training data.
# They are essential for processing the live video frames in the same way.

def normalize_landmarks(landmarks):
    landmarks_np = np.array(landmarks)
    origin = landmarks_np[0].copy()
    landmarks_np -= origin
    max_val = np.max(np.abs(landmarks_np))
    if max_val > 0:
        landmarks_np /= max_val
    return landmarks_np

def draw_landmarks_on_canvas(list_of_hands_landmarks, image_size, connections):
    img = np.full((image_size, image_size, 3), 255, dtype=np.uint8) # White background
    for landmarks in list_of_hands_landmarks:
        center_x, center_y = image_size // 2, image_size // 2
        scale = image_size * 0.35 
        
        points = []
        for lm in landmarks:
            x = int(center_x + lm[0] * scale)
            y = int(center_y + lm[1] * scale)
            points.append((x, y))

        if connections: # Draw connections (skeleton) in Red
            for connection in connections:
                start_idx, end_idx = connection
                if start_idx < len(points) and end_idx < len(points):
                    cv2.line(img, points[start_idx], points[end_idx], (0, 0, 255), 2)
        
        for point in points: # Draw landmarks (joints) in Blue
            cv2.circle(img, point, 4, (255, 0, 0), -1)
            
    return img

# --- Step 4: Main Real-Time Prediction Logic ---

def run_realtime_prediction():
    """
    Captures video from the webcam, processes it, and displays real-time
    gesture predictions in the notebook output.
    """
    # Load the trained YOLOv8 model
    try:
        model = YOLO(MODEL_PATH)
        class_names = model.names
        print("✅ Model loaded successfully.")
        print(f"Available classes: {list(class_names.values())}")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Please ensure the MODEL_PATH is correct.")
        return

    # Initialize MediaPipe Hands
    mp_hands = mp.solutions.hands
    mp_drawing_connections = mp.solutions.hands.HAND_CONNECTIONS
    hands = mp_hands.Hands(
        static_image_mode=False, # Use video mode
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    # Start webcam capture
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ Error: Could not open webcam.")
        return

    # Create a placeholder for the video feed in the notebook output
    display_handle = display(Image(data=b''), display_id=True)
    
    print("\n🚀 Starting real-time prediction... (Interrupt the kernel to stop)")
    
    try:
        while True:
            # Read a frame from the webcam
            ret, frame = cap.read()
            if not ret:
                print("⚠️ Warning: Could not read frame from webcam. Exiting.")
                break

            # Flip the frame horizontally for a more intuitive "mirror" view
            frame = cv2.flip(frame, 1)
            
            # --- The Prediction Pipeline ---
            # 1. Detect landmarks
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb_frame)
            
            prediction_text = "No Hand Detected"

            # 2. If landmarks are found, process them for the model
            if results.multi_hand_landmarks:
                # 2a. Normalize landmarks
                all_hands_landmarks = []
                # This logic is identical to the static image prediction
                if len(results.multi_hand_landmarks) == 2:
                    hand1_lms = np.array([[lm.x, lm.y, lm.z] for lm in results.multi_hand_landmarks[0].landmark])
                    hand2_lms = np.array([[lm.x, lm.y, lm.z] for lm in results.multi_hand_landmarks[1].landmark])
                    center_point = (hand1_lms[0] + hand2_lms[0]) / 2
                    combined_lms = np.vstack([hand1_lms, hand2_lms])
                    combined_lms -= center_point
                    max_val = np.max(np.abs(combined_lms))
                    if max_val > 0: combined_lms /= max_val
                    all_hands_landmarks.append(combined_lms[:21])
                    all_hands_landmarks.append(combined_lms[21:])
                else:
                    hand_lms = [[lm.x, lm.y, lm.z] for lm in results.multi_hand_landmarks[0].landmark]
                    normalized_lms = normalize_landmarks(hand_lms)
                    all_hands_landmarks.append(normalized_lms)
                
                # 2b. Render landmarks to a canvas (this is what the model sees)
                processed_image = draw_landmarks_on_canvas(
                    all_hands_landmarks,
                    IMAGE_SIZE,
                    mp_drawing_connections
                )

                # 2c. Predict using the YOLO model
                yolo_results = model.predict(source=processed_image, verbose=False)
                probs = yolo_results[0].probs.data
                confidence = torch.max(probs).item()
                
                # 2d. Format the prediction text
                if confidence > CONFIDENCE_THRESHOLD:
                    predicted_index = torch.argmax(probs).item()
                    predicted_class = class_names[predicted_index]
                    prediction_text = f"{predicted_class} ({confidence:.2%})"
                else:
                    prediction_text = "Uncertain"
            
            # 3. Display the prediction on the original frame
            cv2.putText(frame, prediction_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3, cv2.LINE_AA)

            # 4. Update the image in the notebook's output cell
            _, jpg_data = cv2.imencode('.jpeg', frame)
            display_handle.update(Image(data=jpg_data.tobytes()))

    except KeyboardInterrupt:
        print("\n⏹️ Prediction stopped by user.")
    finally:
        # Crucial: release the webcam and cleanup
        cap.release()
        hands.close()
        # Clear the output to hide the last frame
        # display_handle.update(None) # Optional: uncomment to clear the image on stop

# --- Step 5: Run the main function ---
run_realtime_prediction()